# Kontrata — Qwen2.5-1.5B LoRA fine-tune

Bu defter **Google Colab GPU** (T4 veya üzeri) içindir. Yerel Mac (8 GB RAM) eğitim için yeterli değildir.

Çalıştırmadan önce:
1. Runtime → Change runtime type → GPU
2. 🔑 simgesinden `HF_TOKEN` secret'ı ekleyin (Hugging Face yazma yetkisi). Jeton koda gömülmez.
3. `ml/data/train.jsonl` ve `val.jsonl` yoksa yerelde `python generate.py --seed 42` çalıştırıp hücre 2'de yükleyin; veya `VERI_KAYNAGI = "github"` ile depodan üretin.

Çıktı:
- Adapter: `oz-fatma/kontrata-qwen-lora-v1`
- Endpoint için birleşik model: `oz-fatma/kontrata-qwen-merged-v1`


In [ ]:
# Hücre 1 — kurulum
# Taban paketler. Jeton yalnızca Colab secrets'tan okunur.

%pip install -q transformers peft trl bitsandbytes accelerate datasets huggingface_hub jsonschema

import subprocess
import sys

print("=== GPU ===")
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError("nvidia-smi başarısız. Runtime → Change runtime type → T4 GPU seçin.")

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA yok. Colab'da GPU çalışma zamanı açın.")
print(f"cuda: {torch.cuda.get_device_name(0)}")

from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Colab Secrets'a HF_TOKEN ekleyin; koda yapıştırmayın.")
login(token=HF_TOKEN)
print("Hugging Face oturumu açıldı (jeton yazdırılmadı).")


In [ ]:
# Hücre 2 — veri
# Yol A: files.upload ile train.jsonl + val.jsonl
# Yol B: GitHub klonu + generate.py (jsonl sürüme girmez)

import json
import subprocess
import sys
import urllib.request
from pathlib import Path

from datasets import Dataset
from google.colab import files

# "upload" veya "github"
VERI_KAYNAGI = "upload"

REPO_CLONE = Path("/content/kontrata")
SCHEMA_URL = "https://raw.githubusercontent.com/oz-fatma/kontrata/main/ml/schema/kontrat.json"
CONTENT = Path("/content")
TRAIN_PATH = CONTENT / "train.jsonl"
VAL_PATH = CONTENT / "val.jsonl"
SCHEMA_PATH = CONTENT / "kontrat.json"

SYSTEM_PROMPT = """Sen bir kontenjan sözleşmesi çıkarım motorusun.
Verilen sözleşme metninden şemaya uygun JSON üret. Açıklama, markdown veya ek metin yazma; yalnızca bir JSON nesnesi döndür.

Çekirdek alanlar (hepsi zorunlu):
- donem (object): baslangic (date|null), bitis (date|null), alt_donemler (array, isteğe bağlı) [{ad (string), baslangic (date), bitis (date)}]
- oda_kontenjanlari (array, en az 1): [{oda_tipi (string; örn. standart, suit, balayi, engelli, aile, deluxe), adet (integer), aciklama (string, isteğe bağlı)}]
- fiyatlar (array, en az 1): [{oda_tipi (string), tutar (number), birim (enum: oda_gecelik | kisi_gecelik), pansiyon (enum: RO | BB | HB | FB | AI | belirtilmemis), alt_donem_ad (string, isteğe bağlı)}]
- release (object): gun (integer), kapsam (enum: isim_listesi | kontenjan_iadesi | her_ikisi | belirtilmemis), kaynak_ifade (string, isteğe bağlı)
- stop_sale (array; boş olabilir): [{baslangic (date), bitis (date), kapsam (string, isteğe bağlı), bildirim_yontemi (enum: yazili | faks | eposta | sistem | belirtilmemis), kaynak_ifade (string, isteğe bağlı)}]

İsteğe bağlı üst alanlar: meta, opsiyonel, cikarim_meta. Çıkaramadığın değeri uydurma; tarih yoksa null kullan.
Tarihler ISO-8601 (YYYY-MM-DD). Sayılar ham sayı olsun.
"""


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def ensure_schema() -> Path:
    if SCHEMA_PATH.exists():
        return SCHEMA_PATH
    cloned = REPO_CLONE / "ml" / "schema" / "kontrat.json"
    if cloned.exists():
        SCHEMA_PATH.write_text(cloned.read_text(encoding="utf-8"), encoding="utf-8")
        return SCHEMA_PATH
    urllib.request.urlretrieve(SCHEMA_URL, SCHEMA_PATH)
    return SCHEMA_PATH


if VERI_KAYNAGI == "upload":
    print("train.jsonl ve val.jsonl seçin (ml/data/ altında üretilmiş dosyalar).")
    uploaded = files.upload()
    names = {Path(n).name: n for n in uploaded}
    if "train.jsonl" not in names or "val.jsonl" not in names:
        raise RuntimeError("Hem train.jsonl hem val.jsonl yüklenmeli.")
    if names["train.jsonl"] != "train.jsonl":
        Path(names["train.jsonl"]).replace(TRAIN_PATH)
    if names["val.jsonl"] != "val.jsonl":
        Path(names["val.jsonl"]).replace(VAL_PATH)
elif VERI_KAYNAGI == "github":
    # jsonl gitignore'dadır; depo klonlanır ve generate.py çalışır.
    if not REPO_CLONE.exists():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/oz-fatma/kontrata.git",
                str(REPO_CLONE),
            ]
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faker"])
    subprocess.check_call(
        [sys.executable, str(REPO_CLONE / "ml" / "generate.py"), "--seed", "42"]
    )
    TRAIN_PATH = REPO_CLONE / "ml" / "data" / "train.jsonl"
    VAL_PATH = REPO_CLONE / "ml" / "data" / "val.jsonl"
    SCHEMA_PATH = REPO_CLONE / "ml" / "schema" / "kontrat.json"
else:
    raise RuntimeError('VERI_KAYNAGI "upload" veya "github" olmalı')

ensure_schema()
train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)
print(f"train={len(train_rows)} val={len(val_rows)}")


def to_messages(row: dict) -> dict:
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["metin"]},
            {
                "role": "assistant",
                "content": json.dumps(row["cikti"], ensure_ascii=False),
            },
        ]
    }


train_ds = Dataset.from_list(train_rows).map(
    to_messages, remove_columns=list(train_rows[0].keys())
)
val_ds = Dataset.from_list(val_rows).map(
    to_messages, remove_columns=list(val_rows[0].keys())
)
print(train_ds[0]["messages"][0]["role"], "sistem talimatı hazır")


In [ ]:
# Hücre 3 — model
# Qwen2.5-1.5B-Instruct, 4-bit nf4 + double quant, LoRA r=16

import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "/content/kontrata-qwen-lora"
MERGED_DIR = "/content/kontrata-qwen-merged"

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print(f"taban: {BASE_MODEL}")
print(f"dtype: {compute_dtype}, LoRA r=16 alpha=32 dropout=0.05")


In [ ]:
# Hücre 4 — eğitim
# SFTTrainer: 3 epoch, batch 4, grad acc 4, lr 2e-4, cosine, warmup 0.03, max_seq 2048

import inspect
import time

from trl import SFTConfig, SFTTrainer

MAX_SEQ_LENGTH = 2048


def pick_kw(params, mapping: dict) -> dict:
    out = {}
    for preferred, value in mapping.items():
        if preferred in params:
            out[preferred] = value
        elif preferred == "max_length" and "max_seq_length" in params:
            out["max_seq_length"] = value
        elif preferred == "eval_strategy" and "evaluation_strategy" in params:
            out["evaluation_strategy"] = value
    return out


sft_params = inspect.signature(SFTConfig.__init__).parameters
sft_kwargs = {
    "output_dir": ADAPTER_DIR,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "logging_steps": 5,
    "save_strategy": "epoch",
    "report_to": "none",
    "optim": "paged_adamw_8bit",
    "gradient_checkpointing": True,
    "bf16": compute_dtype == torch.bfloat16,
    "fp16": compute_dtype != torch.bfloat16,
}
sft_kwargs.update(
    pick_kw(
        sft_params,
        {
            "max_length": MAX_SEQ_LENGTH,
            "eval_strategy": "epoch",
        },
    )
)
if "gradient_checkpointing_kwargs" in sft_params:
    sft_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

try:
    sft_args = SFTConfig(**sft_kwargs)
except TypeError:
    sft_kwargs.pop("max_length", None)
    sft_kwargs.pop("eval_strategy", None)
    sft_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
    sft_kwargs["evaluation_strategy"] = "epoch"
    sft_args = SFTConfig(**sft_kwargs)

trainer_params = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = {
    "model": model,
    "args": sft_args,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
    "peft_config": peft_config,
}
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

t0 = time.perf_counter()
train_result = trainer.train()
elapsed = time.perf_counter() - t0
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

history = trainer.state.log_history
train_pts = [(h.get("step"), h["loss"]) for h in history if "loss" in h]
eval_pts = [(h.get("epoch"), h["eval_loss"]) for h in history if "eval_loss" in h]

print(f"eğitim süresi: {elapsed / 60:.2f} dakika ({elapsed:.0f} sn)")
print(f"son train loss: {train_result.training_loss:.4f}")
print("epoch sonu val kaybı:")
for epoch, loss in eval_pts:
    print(f"  epoch={epoch:.2f}  eval_loss={loss:.4f}")

print("kayıp eğrisi (adım, train_loss):")
for step, loss in train_pts:
    print(f"  step={step}  loss={loss:.4f}")

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    if train_pts:
        axes[0].plot([p[0] for p in train_pts], [p[1] for p in train_pts])
    axes[0].set_xlabel("adım")
    axes[0].set_ylabel("train loss")
    axes[0].set_title("eğitim kaybı")
    if eval_pts:
        axes[1].plot([p[0] for p in eval_pts], [p[1] for p in eval_pts], marker="o")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("eval_loss")
    axes[1].set_title("doğrulama kaybı")
    fig.tight_layout()
    plt.show()
except Exception as e:
    print(f"grafik çizilemedi: {type(e).__name__}")


In [ ]:
# Hücre 5 — hızlı değerlendirme
# val setinden 20 örnek; geçerli JSON, şema uyumu, alan doğruluğu

import json
import re
from collections import OrderedDict
from pathlib import Path

from jsonschema import Draft202012Validator, FormatChecker

CORE_FIELDS = ("donem", "oda_kontenjanlari", "fiyatlar", "release", "stop_sale")
N_EVAL = 20


def extract_json(text: str):
    if not text or not str(text).strip():
        return None
    raw = str(text).strip()
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", raw, re.IGNORECASE)
    if fence:
        raw = fence.group(1).strip()
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(raw[start : end + 1])
    except json.JSONDecodeError:
        return None


def _sort_key(item: dict) -> str:
    return json.dumps(item, sort_keys=True, ensure_ascii=False)


def canonicalize_field(name: str, value):
    if value is None:
        return None
    if name == "donem" and isinstance(value, dict):
        alts = []
        for a in value.get("alt_donemler") or []:
            if isinstance(a, dict):
                alts.append(
                    {"ad": a.get("ad"), "baslangic": a.get("baslangic"), "bitis": a.get("bitis")}
                )
        alts.sort(key=_sort_key)
        return {"baslangic": value.get("baslangic"), "bitis": value.get("bitis"), "alt_donemler": alts}
    if name == "oda_kontenjanlari" and isinstance(value, list):
        rows = [
            {"oda_tipi": x.get("oda_tipi"), "adet": x.get("adet")}
            for x in value
            if isinstance(x, dict)
        ]
        rows.sort(key=_sort_key)
        return rows
    if name == "fiyatlar" and isinstance(value, list):
        rows = []
        for x in value:
            if not isinstance(x, dict):
                continue
            tutar = x.get("tutar")
            if isinstance(tutar, (int, float)):
                tutar = float(tutar)
            rows.append(
                {
                    "oda_tipi": x.get("oda_tipi"),
                    "tutar": tutar,
                    "birim": x.get("birim"),
                    "pansiyon": x.get("pansiyon") or "belirtilmemis",
                    "alt_donem_ad": x.get("alt_donem_ad"),
                }
            )
        rows.sort(key=_sort_key)
        return rows
    if name == "release" and isinstance(value, dict):
        return {"gun": value.get("gun"), "kapsam": value.get("kapsam") or "belirtilmemis"}
    if name == "stop_sale" and isinstance(value, list):
        rows = [
            {
                "baslangic": x.get("baslangic"),
                "bitis": x.get("bitis"),
                "kapsam": x.get("kapsam"),
                "bildirim_yontemi": x.get("bildirim_yontemi") or "belirtilmemis",
            }
            for x in value
            if isinstance(x, dict)
        ]
        rows.sort(key=_sort_key)
        return rows
    return value


def field_matches(gold, pred) -> dict:
    gold_obj = gold if isinstance(gold, dict) else {}
    pred_obj = pred if isinstance(pred, dict) else {}
    return {
        name: canonicalize_field(name, gold_obj.get(name))
        == canonicalize_field(name, pred_obj.get(name))
        for name in CORE_FIELDS
    }


schema = json.loads(Path(SCHEMA_PATH).read_text(encoding="utf-8"))
validator = Draft202012Validator(schema, format_checker=FormatChecker())

model.eval()
model.config.use_cache = True
device = next(model.parameters()).device
sample = val_rows[:N_EVAL]
records = []

for i, row in enumerate(sample):
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["metin"]},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
    parsed = extract_json(gen)
    valid_json = isinstance(parsed, dict)
    schema_ok = valid_json and not any(validator.iter_errors(parsed))
    matches = field_matches(row["cikti"], parsed if valid_json else None)
    records.append({"valid_json": valid_json, "schema_ok": schema_ok, "field_match": matches})
    print(f"{i + 1}/{N_EVAL} json={valid_json} sema={schema_ok}")


def rate(vals):
    return sum(1 for v in vals if v) / len(vals) if vals else 0.0


n = len(records)
summary = OrderedDict(
    [
        ("n", n),
        ("geçerli JSON", rate([r["valid_json"] for r in records])),
        ("şema uyum", rate([r["schema_ok"] for r in records])),
    ]
)
for name in CORE_FIELDS:
    summary[name] = rate([r["field_match"][name] for r in records])

headers = list(summary.keys())
values = [str(summary["n"])] + [f"{100 * v:5.1f}%" for v in list(summary.values())[1:]]
widths = [max(len(h), len(v)) for h, v in zip(headers, values)]
print()
print("  ".join(h.ljust(w) for h, w in zip(headers, widths)))
print("  ".join("-" * w for w in widths))
print("  ".join(v.ljust(w) for v, w in zip(values, widths)))


In [ ]:
# Hücre 6 — yayınlama
# Adapter ve merge edilmiş taban ayrı HF repolarına gider. Endpoint merged sürümü kullanır.

import gc
from pathlib import Path

from huggingface_hub import HfApi
from peft import PeftModel
from transformers import AutoModelForCausalLM

ADAPTER_REPO = "oz-fatma/kontrata-qwen-lora-v1"
MERGED_REPO = "oz-fatma/kontrata-qwen-merged-v1"

ADAPTER_CARD = """---
base_model: Qwen/Qwen2.5-1.5B-Instruct
library_name: peft
license: apache-2.0
tags:
- lora
- qwen2.5
- text-generation
---

# kontrata-qwen-lora-v1

Qwen2.5-1.5B-Instruct üzerine LoRA (r=16, alpha=32) ile eğitilmiş adapter.
HuggingFace Inference Endpoint dağıtımı için **birleşik** repo kullanın: `oz-fatma/kontrata-qwen-merged-v1`.

## Amaç

Otel–acente kontenjan sözleşmesi düz yazısından `kontrat.json` şemasına uygun JSON üretmek
(Okuyucu agent). Sözleşme verisi tesisten çıkmaz; bu ağırlık yalnızca çıkarım motorudur.

## Taban model

`Qwen/Qwen2.5-1.5B-Instruct` — 4-bit nf4 (double quant) üzerinde fine-tune.

## Eğitim verisi

Sentetik, şablonla üretilmiş **320** eğitim + 80 doğrulama örneği (`ml/generate.py --seed 42`).
MEGEP Argos gerçek örneği eğitim kümesine alınmamıştır.

## Bilinen sınırlamalar

- Gerçek operatör kontratı (madde sırası, "normal oda" / "tek kişilik" uyuşmazlığı) sentetik
  şablonlardan sapar; Argos örneği ayrı tutulur ve `evaluate.py` ile ölçülür.
- `opsiyonel` alanlar (iptal, çocuk politikası, ödeme) Aşama 7 kapsamı dışındadır.
- Uzun ekler 2048 tokende kesilir.
- Halüsinasyon: metinde olmayan tarih/tutar uydurulabilir; jsonschema geçerli çıktı gerçeği garanti etmez.
"""

MERGED_CARD = """---
base_model: Qwen/Qwen2.5-1.5B-Instruct
library_name: transformers
license: apache-2.0
tags:
- qwen2.5
- text-generation
---

# kontrata-qwen-merged-v1

`kontrata-qwen-lora-v1` adapter'ının `Qwen/Qwen2.5-1.5B-Instruct` ile birleştirilmiş hali.
HuggingFace Inference Endpoint bu repoyu kullanır.

## Amaç

Kontenjan sözleşmesi metninden şemaya uygun JSON çıkarımı.

## Taban model

`Qwen/Qwen2.5-1.5B-Instruct`

## Eğitim verisi

Sentetik 320 örnek. MEGEP Argos örneği eğitimde yoktur.

## Bilinen sınırlamalar

Adapter kartındaki sınırlamalar geçerlidir. Birleşik ağırlık 4-bit eğitimden gelir; kalite
tam kesinlik LoRA ile aynı kabul edilmemelidir. Üretimde maskeleme katmanı modelden önce çalışır.
"""

(Path(ADAPTER_DIR) / "README.md").write_text(ADAPTER_CARD, encoding="utf-8")
api = HfApi()
print(f"push adapter → {ADAPTER_REPO}")
api.create_repo(ADAPTER_REPO, exist_ok=True, token=HF_TOKEN)
trainer.model.push_to_hub(ADAPTER_REPO, token=HF_TOKEN, commit_message="LoRA adapter v1")
tokenizer.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=str(Path(ADAPTER_DIR) / "README.md"),
    path_in_repo="README.md",
    repo_id=ADAPTER_REPO,
    token=HF_TOKEN,
)

del model
del trainer
gc.collect()
torch.cuda.empty_cache()

print("adapter birleştiriliyor (CPU, 4-bit değil)")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
)
peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = peft_model.merge_and_unload()
Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
(Path(MERGED_DIR) / "README.md").write_text(MERGED_CARD, encoding="utf-8")

print(f"push merged → {MERGED_REPO}")
api.create_repo(MERGED_REPO, exist_ok=True, token=HF_TOKEN)
merged.push_to_hub(MERGED_REPO, token=HF_TOKEN, commit_message="Merged Qwen 1.5B v1")
tokenizer.push_to_hub(MERGED_REPO, token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=str(Path(MERGED_DIR) / "README.md"),
    path_in_repo="README.md",
    repo_id=MERGED_REPO,
    token=HF_TOKEN,
)
print("tamam")
